In [ ]:
"""
Hyeonseok Hwang
hwanghsne@gmail.com

The code is organized into four sections and is designed to be executed in a Jupyter Notebook (.ipynb):
Consider using LLMs for understanding.

1. Import required libraries
2. Data loading and hyperparameter configuration
3. Function definitions
4. Model training and evaluation (main execution)
"""
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sksurv.metrics import concordance_index_censored
import matplotlib.pyplot as plt
from scipy.stats import uniform, randint
import cloudpickle
from xgbse import XGBSEDebiasedBCE
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LogisticRegression
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from sklearn.metrics import roc_curve, auc, classification_report

In [ ]:
# Initial hyperparameters and data loading information.
import os
from dotenv import load_dotenv
load_dotenv()

rng = np.random.RandomState(42)
LOAD_DATA_PATH = os.environ['LOAD_DATA_PATH']
BEST_MODEL_SAVE = os.getenv('BEST_MODEL_SAVE', './best_model.pkl')
CALIBRATION_BEST_SAVE = os.getenv('CALIBRATION_BEST_SAVE', './best_calibration.pkl')
ISOTONIC_SCALERS_SAVE = os.getenv('ISOTONIC_SCALERS_SAVE', './isotonic_scalers.pkl')

# Feature selection
features = [
    'age', 'gender',
    'BMI_value', 'SBP_value', 'smoking', 'hhypertension', 'ddiabetes',
    'ddyslipidemia', 'ischemic_heart_disease', 'CVA',
    'retinopathy', 'LC', 'malignancy', 'Hb_value', 'WBC_value', 'Glucose_value',
    'ethnicity_1', 'ethnicity_2', 'race_1', 'race_2', 'race_3', 'race_4', 'race_5', 'race_6',
    'Na_value', 'K_value', 'Cl_value',
    'eGFR_value', 'Proteinuria_Criteria_Met',
    'heart_failure', 'AST_value', 'FH',
]

# XGBSEDebiasedBCE model initialization and baseline training
xgb_params = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'n_estimators': 100,
    'random_state': 42
}

TIME_BINS = np.array([365, 730, 1095, 1460, 1825])

# RandomizedSearchCV
param_distributions = {
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 0.3),
    'reg_alpha': uniform(0, 0.2),
    'reg_lambda': uniform(1, 1.0),
    'n_estimators': randint(100, 500)
}

# Isotonic regression calibration
calib_times = [365, 730, 1095, 1460, 1825]
eps = 1e-15
days = [365, 730, 1095, 1460, 1825]  # 365, 730, 1095, 1460, 1825
days_for_calib_plot = [365, 730, 1095, 1460, 1825]

# Computation of time-specific calibrated probabilities (p_calibrated_t) and Brier score
n_bootstrap = 1000
alpha = 0.05  # 95% CI => 2.5%, 97.5%


In [ ]:
# Function definitions

class XGBSEDebiasedBCEWrapper(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        n_estimators=100,
        random_state=42,
        **kwargs
    ):
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.gamma = gamma
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.kwargs = kwargs
        self.model = None

    def fit(self, X, y, **fit_kwargs):
        xgb_params = {
            'max_depth': self.max_depth,
            'learning_rate': self.learning_rate,
            'subsample': self.subsample,
            'colsample_bytree': self.colsample_bytree,
            'gamma': self.gamma,
            'reg_alpha': self.reg_alpha,
            'reg_lambda': self.reg_lambda,
            'n_estimators': self.n_estimators,
            'random_state': self.random_state
        }
        xgb_params.update(self.kwargs)  

        self.model = XGBSEDebiasedBCE(xgb_params=xgb_params)
        self.model.fit(X, y, **fit_kwargs)  
        return self

    def predict(self, X):
        return self.model.predict(X)

    def get_params(self, deep=True):
        params = {
            'max_depth': self.max_depth,
            'learning_rate': self.learning_rate,
            'subsample': self.subsample,
            'colsample_bytree': self.colsample_bytree,
            'gamma': self.gamma,
            'reg_alpha': self.reg_alpha,
            'reg_lambda': self.reg_lambda,
            'n_estimators': self.n_estimators,
            'random_state': self.random_state
        }
        params.update(self.kwargs)
        return params

    def set_params(self, **params):
            for key, value in params.items():
                setattr(self, key, value)
            self.kwargs.update(params)
            return self

    @property
    def time_bins(self):
        if self.model is None:
            raise AttributeError("The model has not been fitted yet.")
        return self.model.time_bins

def split_ids_70_10_20(ids_array):
    n = len(ids_array)
    train_end = int(0.7 * n)
    calib_end = int(0.8 * n)  # 70% + 10% = 80%
    train_ids = ids_array[:train_end]
    calib_ids = ids_array[train_end:calib_end]
    test_ids = ids_array[calib_end:]
    return train_ids, calib_ids, test_ids

def cindex_scorer(estimator, X_val, y_val):
    val_time = y_val['time']
    val_event = y_val['event']
    surv_df_val = estimator.predict(X_val)
    time_points = [365, 730, 1095, 1460, 1825]
    c_indexes = []
    for t in time_points:
        if t in estimator.time_bins:
            t_adj = t
        else:
            t_adj = min(estimator.time_bins, key=lambda x: abs(x - t))

        try:
            pred_score = surv_df_val[t_adj].values
        except KeyError:
            closest_time = min(estimator.time_bins, key=lambda x: abs(x - t))
            pred_score = surv_df_val[closest_time].values

        risk_score = 1 - pred_score 
        c_index = concordance_index_censored(
            val_event.astype(bool),
            val_time,
            risk_score
        )[0]
        c_indexes.append(c_index)

    return np.mean(c_indexes)

# Time-dependent c-index with 95% CI
def bootstrap_time_dependent_cindex_ipcw(
    y_train_struct,
    y_test_struct,
    y_test_calibrated_risk,
    days,
    n_boot=1000,
    random_state=42
):
    rng = np.random.RandomState(random_state)
    n_test = y_test_struct.shape[0]
    n_times = len(days)

    cindex_boot = np.zeros((n_boot, n_times), dtype=float)

    for b in range(n_boot):
        indices = rng.randint(0, n_test, size=n_test)
        y_test_struct_bs = y_test_struct[indices]
        risk_bs = y_test_calibrated_risk[indices, :]

        for i, tau in enumerate(days):
            risk_t = risk_bs[:, i]
            cindex_res = concordance_index_ipcw(
                y_train_struct,
                y_test_struct_bs,
                risk_t,
                tau=tau
            )
            cindex_boot[b, i] = cindex_res[0]

    mean_cindex = cindex_boot.mean(axis=0)
    ci_lower = np.percentile(cindex_boot, 2.5, axis=0)
    ci_upper = np.percentile(cindex_boot, 97.5, axis=0)

    return mean_cindex, ci_lower, ci_upper

# Time-dependent AUROC with 95% CI
def bootstrap_cumulative_dynamic_auc(
    y_train_struct,
    y_test_struct,
    y_test_calibrated_risk,  # shape=(n_test, n_times)
    time_points,             # [365, 730, ...]
    n_boot=1000,
    random_state=42
):
    rng = np.random.RandomState(random_state)
    n_test = y_test_struct.shape[0]
    n_times = len(time_points)

    auc_boot = np.zeros((n_boot, n_times), dtype=float)

    for b in range(n_boot):
        indices = rng.randint(0, n_test, n_test)

        y_test_struct_bs = y_test_struct[indices]
        risk_bs = y_test_calibrated_risk[indices, :]

        for i, t in enumerate(time_points):
            risk_t = risk_bs[:, i]
            auc_vals, _ = cumulative_dynamic_auc(
                y_train_struct,
                y_test_struct_bs,
                risk_t,
                [t]
            )
            auc_boot[b, i] = auc_vals[0]

    mean_auc = auc_boot.mean(axis=0)
    lower_ci = np.percentile(auc_boot, 2.5, axis=0)
    upper_ci = np.percentile(auc_boot, 97.5, axis=0)

    return mean_auc, lower_ci, upper_ci

def plot_time_specific_roc_curves(
    time_points,  # e.g. [365, 730, 1095, 1460, 1825]
    y_time,       # (n_samples)
    y_event,      # (n_samples)
    y_risk_all_times  # (n_samples, len(time_points))
):
    for i, t in enumerate(time_points):
        pred_risk_t = y_risk_all_times[:, i]  # shape=(n_samples,)

        actual_label = ((y_event == 1) & (y_time <= t)).astype(int)

        fpr, tpr, thresholds = roc_curve(actual_label, pred_risk_t)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='darkblue', linestyle='-', label='ROC curve')
        plt.plot([0, 1], [0, 1], 'k--') 
        plt.title(f'ROC at {t} days')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc='lower right')
        plt.grid(False)
        plt.show()

def calibration_slope_intercept(y_true, y_prob):
    eps = 1e-7
    y_prob_clipped = np.clip(y_prob, eps, 1 - eps)
    logit_prob = np.log(y_prob_clipped / (1 - y_prob_clipped))

    lr = LogisticRegression(fit_intercept=True, solver='lbfgs')
    lr.fit(logit_prob.reshape(-1, 1), y_true)

    slope = lr.coef_[0][0]
    intercept = lr.intercept_[0]
    return slope, intercept

def plot_calibration_curve(y_true, y_prob, title="", n_bins=20):
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="uniform")

    # Calibration Plot
    plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
    plt.plot(mean_pred, frac_pos, marker='o', color='darkblue', linestyle='None', label='Actual')
    plt.title(title)
    plt.xlabel("Predicted Probability")
    plt.ylabel("Observed Frequency")
    plt.legend()
    plt.grid(False)

    slope, intercept = calibration_slope_intercept(y_true, y_prob)
    ece = np.mean(np.abs(frac_pos - mean_pred))
    brier = np.mean((y_true - y_prob)**2)

    return slope, intercept, ece, brier

def evaluate_calibration_over_times(
    days_for_calib_plot,
    y_test_calibrated_risk,
    y_test_struct, results_calib
):
    # fig, axes = plt.subplots(1, len(days_for_calib_plot), figsize=(6*len(days_for_calib_plot), 4))
    fig, axes = plt.subplots(len(days_for_calib_plot), 1, figsize=(8, 5 * len(days_for_calib_plot)))
    for i, tau in enumerate(days_for_calib_plot):
        ax = axes[i]
        plt.sca(ax)

        risk_prob = y_test_calibrated_risk[:, i]

        obs_event = y_test_struct['event']
        obs_time = y_test_struct['time']
        y_label_tau = ((obs_event == True) & (obs_time <= tau)).astype(int)

        slope, intercept, ece, brier = plot_calibration_curve(
            y_label_tau, risk_prob,
            title=f"{tau} days Calibration",
            n_bins=20
        )
        results_calib[tau] = (slope, intercept, ece, brier)

    plt.tight_layout()
    plt.show()
    

def bootstrap_calibration_metrics(
    days_for_calib_plot,
    y_test_calibrated_risk,  # shape=(n_test, len(days_for_calib_plot))
    y_test_struct,           # (dtype=[('event','?'),('time','<f8')]), shape=(n_test,)
    plot_calib_func,         # (y_label, y_prob) -> slope, intercept, ece, brier
    n_boot=1000,
    random_state=42
):
    rng = np.random.RandomState(random_state)
    n_test = len(y_test_calibrated_risk)
    n_times = len(days_for_calib_plot)

    metrics_boot = np.zeros((n_boot, n_times, 4), dtype=float)

    for b in range(n_boot):
        indices = rng.randint(0, n_test, size=n_test)

        y_struct_bs = y_test_struct[indices]
        risk_bs = y_test_calibrated_risk[indices, :]

        for i, tau in enumerate(days_for_calib_plot):
            risk_prob = risk_bs[:, i]
            obs_event_bs = y_struct_bs['event']
            obs_time_bs = y_struct_bs['time']
            y_label_bs = ((obs_event_bs == True) & (obs_time_bs <= tau)).astype(int)

            slope_bs, intercept_bs, ece_bs, brier_bs = plot_calib_func(y_label_bs, risk_prob, n_bins=20)
            metrics_boot[b, i, 0] = slope_bs
            metrics_boot[b, i, 1] = intercept_bs
            metrics_boot[b, i, 2] = ece_bs
            metrics_boot[b, i, 3] = brier_bs

    mean_metrics = {}
    ci_lower = {}
    ci_upper = {}

    for i, tau in enumerate(days_for_calib_plot):
        # metrics_boot[:, i, :] => shape=(n_boot,4)
        boot_slope = metrics_boot[:, i, 0]
        boot_intercept = metrics_boot[:, i, 1]
        boot_ece = metrics_boot[:, i, 2]
        boot_brier = metrics_boot[:, i, 3]

        mean_slope = np.mean(boot_slope)
        mean_intercept = np.mean(boot_intercept)
        mean_ece = np.mean(boot_ece)
        mean_brier = np.mean(boot_brier)

        l_slope, u_slope = np.percentile(boot_slope, [2.5, 97.5])
        l_intercept, u_intercept = np.percentile(boot_intercept, [2.5, 97.5])
        l_ece, u_ece = np.percentile(boot_ece, [2.5, 97.5])
        l_brier, u_brier = np.percentile(boot_brier, [2.5, 97.5])

        mean_metrics[tau] = (mean_slope, mean_intercept, mean_ece, mean_brier)
        ci_lower[tau] = (l_slope, l_intercept, l_ece, l_brier)
        ci_upper[tau] = (u_slope, u_intercept, u_ece, u_brier)

    return mean_metrics, ci_lower, ci_upper

def simple_calib_func(y_true, y_prob, n_bins=30):
    eps = 1e-12
    p_clipped = np.clip(y_prob, eps, 1-eps)

    x_logit = np.log(p_clipped / (1 - p_clipped))
    X_ = x_logit.reshape(-1, 1)
    y_ = y_true.astype(int)

    logit_model = LogisticRegression(fit_intercept=True, solver='lbfgs')
    logit_model.fit(X_, y_)

    slope = logit_model.coef_[0][0]
    intercept = logit_model.intercept_[0]

    ece = np.mean(np.abs(p_clipped - y_))
    brier = np.mean((p_clipped - y_)**2)

    return slope, intercept, ece, brier

# Classification report at best threshold
def time_specific_classification_report(
    time_points,
    y_time,       # shape=(n_samples,)
    y_event,      # shape=(n_samples,)
    y_risk_all_times  # shape=(n_samples, len(time_points))
):
    for i, t in enumerate(time_points):
        pred_risk_t = y_risk_all_times[:, i]  # shape=(n_samples,)
        actual_label = ((y_event == 1) & (y_time <= t)).astype(int)

        fpr, tpr, thresholds = roc_curve(actual_label, pred_risk_t)
        roc_auc = auc(fpr, tpr)

        # Best threshold
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        best_thr = thresholds[best_idx]

        pred_label = (pred_risk_t >= best_thr).astype(int)

        # Classification Report
        print(f"=== Classification at {t} days ===")
        print(f"Best Threshold = {best_thr:.4f} (Youden's J)")
        print(classification_report(actual_label, pred_label, digits=4))

def permutation_feature_importance_xgbse(
    model,
    X,
    y_time,
    y_event,
    isotonic_scalers=None,
    time_points=None,
    n_repeats=20,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    baseline_score = calc_avg_cindex_xgbse_calibrated(
        model=model,
        X=X,
        y_time=y_time,
        y_event=y_event,
        isotonic_scalers=isotonic_scalers,
        time_points=time_points
    )

    importance_dict = {}

    for col in X.columns:
        drop_list = []
        for _ in range(n_repeats):
            X_temp = X.copy(deep=True)
            permuted_col = rng.permutation(X_temp[col].values)
            X_temp[col] = permuted_col

            score_perm = calc_avg_cindex_xgbse_calibrated(
                model=model,
                X=X_temp,
                y_time=y_time,
                y_event=y_event,
                isotonic_scalers=isotonic_scalers,
                time_points=time_points
            )

            drop = baseline_score - score_perm
            drop_list.append(drop)

        importance_dict[col] = np.mean(drop_list)

    return baseline_score, importance_dict

# Permutation feature importance (calibrated)
def calc_avg_cindex_xgbse_calibrated(model, X, y_time, y_event, isotonic_scalers, time_points=None):
    if time_points is None:
        time_points = [365, 730, 1095, 1460, 1825]

    surv_df = model.predict(X)
    model_time_bins = model.time_bins

    cindexes = []
    for t in time_points:
        t_adj = min(model_time_bins, key=lambda x: abs(x - t))
        pred_surv = surv_df[t_adj].values  # shape: (n_samples, )
        calibrated_surv = isotonic_scalers[t_adj].predict(pred_surv.reshape(-1, 1))

        risk = 1 - calibrated_surv

        c_idx = concordance_index_censored(
            y_event.astype(bool),
            y_time.values,
            risk
        )[0]
        cindexes.append(c_idx)

    return np.mean(cindexes)


In [ ]:
# Main execution (model training and evaluation)

df_spark_ckd_prediction = spark.read.format("delta").load(LOAD_DATA_PATH)
xgbse_data = df_spark_ckd_prediction.toPandas()

X = xgbse_data[features]
y_time = xgbse_data['time_interval']
y_event = xgbse_data['event'].astype(int)

# Train/Test Split (GroupShuffleSplit by person_id)
stratify_df = xgbse_data.groupby('person_id')['Incident_CKD'].max().reset_index()

pos_ids = stratify_df.loc[stratify_df['Incident_CKD'] == 1, 'person_id'].values
neg_ids = stratify_df.loc[stratify_df['Incident_CKD'] == 0, 'person_id'].values

pos_ids_shuffled = rng.permutation(pos_ids)
neg_ids_shuffled = rng.permutation(neg_ids)

pos_train, pos_calib, pos_test = split_ids_70_10_20(pos_ids_shuffled)
neg_train, neg_calib, neg_test = split_ids_70_10_20(neg_ids_shuffled)

train_person_ids = np.concatenate([pos_train, neg_train])
calib_person_ids = np.concatenate([pos_calib, neg_calib])
test_person_ids = np.concatenate([pos_test, neg_test])

X_train = X[xgbse_data['person_id'].isin(train_person_ids)]
X_calib = X[xgbse_data['person_id'].isin(calib_person_ids)]
X_test  = X[xgbse_data['person_id'].isin(test_person_ids)]

y_train_time = y_time[xgbse_data['person_id'].isin(train_person_ids)]
y_calib_time = y_time[xgbse_data['person_id'].isin(calib_person_ids)]
y_test_time  = y_time[xgbse_data['person_id'].isin(test_person_ids)]

y_train_event = y_event[xgbse_data['person_id'].isin(train_person_ids)]
y_calib_event = y_event[xgbse_data['person_id'].isin(calib_person_ids)]
y_test_event  = y_event[xgbse_data['person_id'].isin(test_person_ids)]

y_train_struct = np.array(
    list(zip(y_train_event.astype(bool), y_train_time)),
    dtype=[('event', '?'), ('time', '<f8')]
)
y_calib_struct = np.array(
    list(zip(y_calib_event.astype(bool), y_calib_time)),
    dtype=[('event', '?'), ('time', '<f8')]
)
y_test_struct = np.array(
    list(zip(y_test_event.astype(bool), y_test_time)),
    dtype=[('event', '?'), ('time', '<f8')]
)

model = XGBSEDebiasedBCEWrapper(
    max_depth=xgb_params['max_depth'],
    learning_rate=xgb_params['learning_rate'],
    subsample=xgb_params['subsample'],
    colsample_bytree=xgb_params['colsample_bytree'],
    gamma=xgb_params['gamma'],
    reg_alpha=xgb_params['reg_alpha'],
    reg_lambda=xgb_params['reg_lambda'],
    n_estimators=xgb_params['n_estimators'],
    random_state=xgb_params['random_state']
)

model.fit(X_train, y_train_struct, time_bins=TIME_BINS)

# GroupKFold CV
cv = GroupKFold(n_splits=5)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=XGBSEDebiasedBCEWrapper(),
    param_distributions=param_distributions,
    n_iter=50,
    scoring=cindex_scorer,
    cv=cv,
    verbose=1,
    n_jobs=1,
    random_state=42,
    error_score='raise'
)

train_groups = xgbse_data.loc[X_train.index, 'person_id']
random_search.fit(X_train, y_train_struct, groups=train_groups)
print("RandomizedSearchCV completed.")

best_params = random_search.best_params_
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"{param}: {value}")
print(f"Best CV Concordance Index: {random_search.best_score_:.4f}")

best_model = random_search.best_estimator_
best_model.fit(X_train, y_train_struct, time_bins=TIME_BINS)

with open(BEST_MODEL_SAVE, 'wb') as f:
    cloudpickle.dump(best_model, f)

isotonic_scalers = {}

for t in calib_times:
    surv_calib = best_model.predict(X_calib)
    closest_bin = min(best_model.time_bins, key=lambda x: abs(x - t))
    p_surv_t_calib = surv_calib[closest_bin].values  
    p_risk_t_calib = 1.0 - p_surv_t_calib          

    p_risk_t_calib_clipped = np.clip(p_risk_t_calib, eps, 1 - eps)

    obs_event_cal = y_calib_struct['event']
    obs_time_cal = y_calib_struct['time']
    observed_t_calib = ((obs_event_cal == True) & (obs_time_cal <= t)).astype(int)

    lr_cal = IsotonicRegression(out_of_bounds='clip')
    lr_cal.fit(p_risk_t_calib_clipped, observed_t_calib)

    isotonic_scalers[t] = lr_cal


y_test_calibrated_risk = np.zeros((X_test.shape[0], len(calib_times)))

for i, t in enumerate(calib_times):
    surv_test = best_model.predict(X_test)
    closest_bin = min(best_model.time_bins, key=lambda x: abs(x - t))
    p_surv_t_test = surv_test[closest_bin].values
    p_risk_t_test = 1.0 - p_surv_t_test

    # clip
    p_risk_t_test_clipped = np.clip(p_risk_t_test, eps, 1 - eps)

    scaler_t = isotonic_scalers[t]
    p_calibrated_t = scaler_t.predict(p_risk_t_test_clipped)

    y_test_calibrated_risk[:, i] = p_calibrated_t

    obs_event_test = y_test_struct['event']
    obs_time_test = y_test_struct['time']
    observed_t_test = ((obs_event_test == True) & (obs_time_test <= t)).astype(int)

    bs_t = brier_score_loss(observed_t_test, p_calibrated_t)  # Brier Score
    cindex_t = concordance_index_censored(
        obs_event_test.astype(bool),
        obs_time_test,
        p_calibrated_t
    )[0]  
    auc_t = roc_auc_score(observed_t_test, p_calibrated_t)

    print(f"\n=== [Isotonic Calibration] t={t} days ===")
    print(f" Brier Score  = {bs_t:.4f}")
    print(f" C-Index      = {cindex_t:.4f}")
    print(f" ROC-AUC      = {auc_t:.4f}")


# Calibration
with open(CALIBRATION_BEST_SAVE, 'wb') as f:
    cloudpickle.dump(isotonic_scalers, f)

# Time-dependent c-index
cindex_results = {}

for i, t in enumerate(days):
    risk_t = y_test_calibrated_risk[:, i]

    cindex_res = concordance_index_ipcw( 
        y_train_struct,
        y_test_struct,
        risk_t,
        tau=t
    )
    cindex_val = cindex_res[0]  # (cindex, concordant, discordant, ...)
    cindex_results[t] = cindex_val
    print(f"{t} days => C-index: {cindex_val:.4f}")

mean_cidx, ci_lo, ci_hi = bootstrap_time_dependent_cindex_ipcw(
    y_train_struct,
    y_test_struct,
    y_test_calibrated_risk,
    days,
    n_boot=500
)

for t, m, l, u in zip(days, mean_cidx, ci_lo, ci_hi):
    print(f"{t} days => C-index: {m:.4f} (95%CI: [{l:.4f}, {u:.4f}])")


# Time-dependent AUROC
for i, t in enumerate(days):
    risk_t = y_test_calibrated_risk[:, i]

    auc_vals, _ = cumulative_dynamic_auc(
        y_train_struct,  
        y_test_struct,   
        risk_t,
        [t]
    )
    print(f"{t} days => AUROC: {auc_vals[0]:.4f}")

mean_auc, ci_lower, ci_upper = bootstrap_cumulative_dynamic_auc(
    y_train_struct,
    y_test_struct,
    y_test_calibrated_risk,
    time_points=[365, 730, 1095, 1460, 1825],
    n_boot=1000
)

for t, m, l, u in zip([365, 730, 1095, 1460, 1825], mean_auc, ci_lower, ci_upper):
    print(f"{t} days => AUC={m:.4f} (95% CI: [{l:.4f}, {u:.4f}])")

time_points = days
plot_time_specific_roc_curves(
    time_points,
    y_test_time,    
    y_test_event,   
    y_test_calibrated_risk  # shape=(n_samples, 5)
)

# Brier score with 95% CI

results_brier = {}

for i, t in enumerate(calib_times):
    p_calibrated_t = y_test_calibrated_risk[:, i]

    obs_event_test = y_test_struct['event']
    obs_time_test = y_test_struct['time']
    observed_t_test = ((obs_event_test == True) & (obs_time_test <= t)).astype(int)

    brier_orig = brier_score_loss(observed_t_test, p_calibrated_t)

    boot_scores = []
    n_test = len(X_test)
    for _ in range(n_bootstrap):
        sample_idx = np.random.choice(n_test, size=n_test, replace=True)
        y_boot = observed_t_test[sample_idx]
        p_boot = p_calibrated_t[sample_idx]

        bs_boot = brier_score_loss(y_boot, p_boot)
        boot_scores.append(bs_boot)

    boot_scores = np.array(boot_scores)
    brier_mean = np.mean(boot_scores)
    ci_lower = np.percentile(boot_scores, 100 * alpha / 2)
    ci_upper = np.percentile(boot_scores, 100 * (1 - alpha / 2))

    results_brier[t] = {
        'brier_original': brier_orig,
        'brier_mean': brier_mean,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }

for t in calib_times:
    b_orig = results_brier[t]['brier_original']
    b_mean = results_brier[t]['brier_mean']
    b_low = results_brier[t]['ci_lower']
    b_high = results_brier[t]['ci_upper']
    print(f"[{t} days] Brier={b_orig:.3f}, "
          f"Bootstrap Mean={b_mean:.3f}, "
          f"95% CI=[{b_low:.3f}, {b_high:.3f}]")


results_calib = {}

evaluate_calibration_over_times(
    days_for_calib_plot=[365, 730, 1095, 1460, 1825],
    y_test_calibrated_risk=y_test_calibrated_risk,
    y_test_struct=y_test_struct, results_calib=results_calib
)

print("=== Calibration Metrics (Slope, Intercept, ECE, Brier) ===")
for tau in days_for_calib_plot:
    slope, intercept, ece, brier = results_calib[tau]
    print(f"[{tau} days] Slope={slope:.2f}, Intercept={intercept:.2f}, "
          f"ECE={ece:.3f}, Brier={brier:.3f}")

# Calibration metrics with 95% CI


mean_metrics, ci_lower, ci_upper = bootstrap_calibration_metrics(
    days_for_calib_plot,
    y_test_calibrated_risk,
    y_test_struct,
    plot_calib_func=simple_calib_func,
    n_boot=1000
)

print("===== Calibration Metrics + 95% CI =====")
for tau in days_for_calib_plot:
    (mslope, mintc, mece, mbrier) = mean_metrics[tau]
    (lslope, lintc, lece, lbrier) = ci_lower[tau]
    (uslope, uintc, uece, ubrier) = ci_upper[tau]
    print(f"[{tau} days]")
    print(f" Slope: {mslope:.2f} (95%CI [{lslope:.2f}, {uslope:.2f}])")
    print(f" Intercept: {mintc:.2f} (95%CI [{lintc:.2f}, {uintc:.2f}])")
    print(f" ECE: {mece:.3f} (95%CI [{lece:.3f}, {uece:.3f}])")
    print(f" Brier: {mbrier:.3f} (95%CI [{lbrier:.3f}, {ubrier:.3f}])")
    print("-----")


time_points = days
time_specific_classification_report(
    time_points,
    y_test_time,    
    y_test_event,      
    y_test_calibrated_risk  # shape=(n_samples, 5)
)

baseline_score, feat_importances = permutation_feature_importance_xgbse(
    model=best_model,
    X=X_test,
    y_time=y_test_time,
    y_event=y_test_event,
    isotonic_scalers=isotonic_scalers,
    time_points=[365, 730, 1095, 1460, 1825],
    n_repeats=20,
    random_state=42
)

print(f"Baseline C-index(avg over time_points) = {baseline_score:.4f}")
sorted_importances = sorted(feat_importances.items(), key=lambda x: x[1], reverse=True)

print("\n=== Permutation Feature Importance (c-index drop) ===")
for col, imp_val in sorted_importances:
    print(f"{col:<30}: {imp_val:.4f}")
